In [ ]:

import json, math, shutil
from pathlib import Path

import cv2
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

# ── font (English only in plots) ──
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.unicode_minus': False,
    'figure.dpi': 100,
})

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

CAPTURES_DIR = Path('/home/vsc/LLM_TUNE/AIC_Sejong/aic_data/captures')
VIEWS = ['left', 'center', 'right']

ALL_SESSIONS = sorted([
    p for p in CAPTURES_DIR.iterdir()
    if p.is_dir() and (p / 'steps.jsonl').exists()
])

def parse_session(d: Path) -> dict:
    parts = d.name.split('_')
    return {'session': d.name, 'task_type': parts[2], 'rail': parts[3], 'session_dir': d}

print(f'{len(ALL_SESSIONS)} valid sessions')


In [ ]:

# ── 1-1: Parse steps.jsonl ───────────────────────────────────────────────────
step_rows = []
for sess_dir in tqdm(ALL_SESSIONS, desc='Parsing steps'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        for line in f:
            d = json.loads(line)
            obs   = d.get('observation', {})
            ext   = d.get('extras', {})
            tfm   = d.get('transforms', {})
            wrench = obs.get('wrist_wrench', {})
            force  = wrench.get('force', {})
            torque = wrench.get('torque', {})
            ctrl   = obs.get('controller_state', {})
            tcp    = ctrl.get('tcp_error', [None]*6)
            plug_t = tfm.get('plug', {}).get('translation', {})
            port_t = tfm.get('port', {}).get('translation', {})
            step_rows.append({
                'session':    info['session'],
                'task_type':  info['task_type'],
                'rail':       info['rail'],
                'step':       d['step'],
                'phase':      d['phase'],
                'tip_x_error': ext.get('tip_x_error'),
                'tip_y_error': ext.get('tip_y_error'),
                'z_offset':   ext.get('z_offset'),
                'force_x': force.get('x'), 'force_y': force.get('y'), 'force_z': force.get('z'),
                'torque_x': torque.get('x'), 'torque_y': torque.get('y'), 'torque_z': torque.get('z'),
                'plug_x': plug_t.get('x'), 'plug_y': plug_t.get('y'), 'plug_z': plug_t.get('z'),
                'port_x': port_t.get('x'), 'port_y': port_t.get('y'), 'port_z': port_t.get('z'),
            })

steps_df = pd.DataFrame(step_rows)
print(f'Total steps: {len(steps_df):,}')


In [ ]:

# ── 1-2: Phase & Task imbalance ──────────────────────────────────────────────
phase_cnt  = steps_df['phase'].value_counts().reindex(['approach','insert','stabilize'])
task_cnt   = steps_df['task_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

phase_cnt.plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868'], rot=0)
for bar, v in zip(axes[0].patches, phase_cnt):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                 f'{v/phase_cnt.sum()*100:.1f}%', ha='center', fontsize=9)
axes[0].set_title('Phase Distribution (step count)')
axes[0].set_ylabel('Steps')

task_cnt.plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452'], rot=0)
for bar, v in zip(axes[1].patches, task_cnt):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f'{v/task_cnt.sum()*100:.1f}%', ha='center', fontsize=9)
axes[1].set_title('Task Type Distribution (step count)')
axes[1].set_ylabel('Steps')

plt.suptitle('Dataset Imbalance  [Input Bias Source]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n[Bias] insert dominates: {:.0f}% of all frames'.format(
    phase_cnt['insert']/phase_cnt.sum()*100))
print('[Bias] nic dominates: {:.0f}% of all frames'.format(
    task_cnt['nic']/task_cnt.sum()*100))


In [ ]:

# ── 2-1: tip_error 2D scatter + convergence over step ────────────────────────
# tip_x/y_error = port.xy - plug.xy  (world frame, metres)
# I-controller converges these to 0 → distribution biased toward 0

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) 2D scatter: approach
for ax, phase, title in zip(
    axes,
    ['approach', 'insert', 'insert'],
    ['Approach: tip XY error', 'Insert: tip XY error', 'Insert: error convergence']
):
    if title.endswith('convergence'):
        for sess, sg in steps_df[steps_df['phase']=='insert'].groupby('session'):
            err_mag = np.sqrt(sg['tip_x_error']**2 + sg['tip_y_error']**2) * 1000
            ax.plot(sg['step'] - sg['step'].min(), err_mag,
                    alpha=0.3, lw=0.7, color='steelblue')
        ax.set_xlabel('Step (within insert phase)')
        ax.set_ylabel('|tip_error| (mm)')
        ax.set_title(title)
    else:
        sub = steps_df[steps_df['phase']==phase]
        ax.scatter(sub['tip_x_error']*1000, sub['tip_y_error']*1000,
                   alpha=0.05, s=2, c=sub['step'], cmap='viridis')
        ax.axhline(0, color='red', lw=0.5)
        ax.axvline(0, color='red', lw=0.5)
        ax.set_xlabel('tip_x_error (mm)')
        ax.set_ylabel('tip_y_error (mm)')
        ax.set_title(title)
        ax.set_aspect('equal')

plt.suptitle('Output Bias: tip error distribution  [Output Space]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

ins = steps_df[steps_df['phase']=='insert']
print('\ntip_error stats (insert phase):')
display(ins.groupby('task_type')[['tip_x_error','tip_y_error']].describe().round(5))


In [ ]:

# ── 2-2: z_offset trajectory & force_z distribution ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# z_offset per session
for task, ax in zip(['nic','sc'], axes[:2]):
    for sess, sg in steps_df[steps_df['task_type']==task].groupby('session'):
        ax.plot(sg['step'], sg['z_offset'], alpha=0.35, lw=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('z_offset (m)')
    ax.set_title(f'z_offset trajectory  [{task.upper()}]')

# force_z by phase
sns.histplot(data=steps_df, x='force_z', hue='phase', bins=50,
             stat='density', common_norm=False, element='step', fill=False, ax=axes[2])
axes[2].set_title('force_z distribution by phase')
axes[2].set_xlabel('force_z (N)')

plt.suptitle('Output Bias: trajectory & contact force  [Output Space]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

# ── 3-1: Image sampling ──────────────────────────────────────────────────────
RESIZE_H, RESIZE_W = 256, 288
N_PER_SESSION = 30

def load_gray(path, h=RESIZE_H, w=RESIZE_W):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32)

def load_rgb(path, h=RESIZE_H, w=RESIZE_W):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)

# build image path table
img_rows = []
for sess_dir in ALL_SESSIONS:
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    idxs = np.linspace(0, len(lines)-1, N_PER_SESSION, dtype=int)
    for i in idxs:
        d = json.loads(lines[i])
        obs = d.get('observation', {})
        for v in VIEWS:
            k = f'{v}_image'
            if k in obs:
                img_rows.append({
                    'session': info['session'], 'task_type': info['task_type'],
                    'rail': info['rail'], 'view': v, 'phase': d['phase'],
                    'step': d['step'], 'path': Path(obs[k]['path']),
                })

img_df = pd.DataFrame(img_rows)
print(f'Sampled image records: {len(img_df):,}')


In [ ]:

# ── 3-2: Spatial variance heatmap — where does motion concentrate? ────────────
# High variance pixel = robot/plug passes through that location frequently
# If concentrated → spatial input bias

var_maps = {}
for (view, task), sub in tqdm(img_df.groupby(['view','task_type']), desc='Variance maps'):
    frames = []
    for path in sub['path']:
        try: frames.append(load_gray(path))
        except: continue
    if len(frames) > 1:
        var_maps[(view, task)] = np.stack(frames).var(axis=0)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for row_i, task in enumerate(['nic','sc']):
    for col_i, view in enumerate(VIEWS):
        ax = axes[row_i][col_i]
        vm = var_maps.get((view, task))
        if vm is None: continue
        im = ax.imshow(vm, cmap='hot')
        plt.colorbar(im, ax=ax, fraction=0.046)
        hy, hx = np.unravel_index(vm.argmax(), vm.shape)
        ax.scatter([hx],[hy], c='cyan', s=60, marker='+', linewidths=2)
        ax.set_title(f'{task.upper()} — {view}  peak=({hx},{hy})')
        ax.axis('off')

plt.suptitle('Pixel Variance Heatmap  [Input Spatial Bias]', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Spatial concentration: what fraction of pixels hold 50% of variance energy?
def spatial_conc(vm, thr=0.5):
    flat = np.sort(vm.flatten())[::-1]
    cs = np.cumsum(flat)
    n = np.searchsorted(cs, cs[-1]*thr) + 1
    return n / flat.size

print('\nSpatial concentration (fraction of pixels holding 50% of variance energy):')
print(f'  {"view":8s} {"task":5s}  value   note')
for (view, task), vm in sorted(var_maps.items()):
    c = spatial_conc(vm)
    flag = 'BIASED' if c < 0.15 else ('OK' if c > 0.30 else 'caution')
    print(f'  {view:8s} {task:5s}  {c:.3f}   {flag}')


In [ ]:

# ── 3-3: 8x8 grid variance heatmap — which grid cells are active? ─────────────
GRID = 8
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for row_i, task in enumerate(['nic','sc']):
    for col_i, view in enumerate(VIEWS):
        ax = axes[row_i][col_i]
        vm = var_maps.get((view, task))
        if vm is None: continue
        H, W = vm.shape
        gh, gw = H//GRID, W//GRID
        grid = np.array([[vm[i*gh:(i+1)*gh, j*gw:(j+1)*gw].mean()
                          for j in range(GRID)] for i in range(GRID)])
        grid /= grid.max() + 1e-8
        sns.heatmap(grid, ax=ax, cmap='YlOrRd', annot=True, fmt='.2f',
                    cbar=False, linewidths=0.3, vmin=0, vmax=1)
        ax.set_title(f'{task.upper()} — {view}')

plt.suptitle(f'{GRID}x{GRID} Grid Variance  [Input Spatial Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

# ── 4-1: CV feature functions ────────────────────────────────────────────────
def fft_high_ratio(gray, frac=0.18):
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
    mag = np.abs(f)
    h, w = gray.shape
    cy, cx = h//2, w//2
    ry, rx = int(h*frac/2), int(w*frac/2)
    mask = np.zeros_like(gray, dtype=bool)
    mask[cy-ry:cy+ry+1, cx-rx:cx+rx+1] = True
    return float(mag[~mask].sum() / (mag.sum()+1e-6))

def hue_entropy(hsv, bins=36):
    sat = hsv[:,:,1]
    hue = hsv[:,:,0]
    valid = sat > 25
    if valid.sum() < 64: return np.nan
    hist, _ = np.histogram(hue[valid], bins=bins, range=(0,180), density=True)
    hist = hist[hist>0]
    return float(-(hist * np.log(hist+1e-9)).sum())

def fg_bbox(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    h, w = lab.shape[:2]
    border = np.concatenate([lab[:16].reshape(-1,3), lab[-16:].reshape(-1,3),
                              lab[:,:16].reshape(-1,3), lab[:,-16:].reshape(-1,3)])
    dist = np.linalg.norm(lab.astype(np.float32) - np.median(border,0).astype(np.float32), axis=2)
    mask = ((dist > np.percentile(dist,82))*255).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((5,5),np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9,9),np.uint8))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    ic = np.array([w/2, h/2])
    best, bscore = None, None
    for i in range(1, n):
        a = stats[i, cv2.CC_STAT_AREA]
        if a < 80: continue
        cx = stats[i, cv2.CC_STAT_LEFT] + stats[i, cv2.CC_STAT_WIDTH]/2
        cy = stats[i, cv2.CC_STAT_TOP]  + stats[i, cv2.CC_STAT_HEIGHT]/2
        s = a - 3000*np.linalg.norm((np.array([cx,cy])-ic)/np.array([w,h]))
        if bscore is None or s > bscore: bscore, best = s, i
    if best is None: return np.nan, np.nan
    s = stats[best]
    return (s[cv2.CC_STAT_LEFT]+s[cv2.CC_STAT_WIDTH]/2)/w, \
           (s[cv2.CC_STAT_TOP] +s[cv2.CC_STAT_HEIGHT]/2)/h

print('CV functions ready')


In [ ]:

# ── 4-2: CV feature extraction (view x task_type x phase) ────────────────────
N_PER_PHASE = 12
cv_rows = []

for sess_dir in tqdm(ALL_SESSIONS, desc='CV features'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        all_lines = f.readlines()
    phase_groups = {'approach':[], 'insert':[]}
    for ln in all_lines:
        d = json.loads(ln)
        if d['phase'] in phase_groups:
            phase_groups[d['phase']].append(d)

    for ph, records in phase_groups.items():
        step = max(1, len(records)//N_PER_PHASE)
        for d in records[::step][:N_PER_PHASE]:
            obs = d.get('observation', {})
            for view in VIEWS:
                key = f'{view}_image'
                if key not in obs: continue
                try:
                    rgb  = load_rgb(Path(obs[key]['path']))
                    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
                    gf   = gray.astype(np.float32)
                    hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
                    val, sat = hsv[:,:,2], hsv[:,:,1]

                    lap  = float(cv2.Laplacian(gf, cv2.CV_32F).var())
                    ns   = float((gf - cv2.GaussianBlur(gf,(0,0),1.0)).std())
                    fft  = fft_high_ratio(gray)
                    edge = float((cv2.Canny(gray,40,120)>0).mean())
                    bri  = float(val.mean()/255)
                    sat_m= float(sat.mean()/255)
                    hent = hue_entropy(hsv)
                    fx, fy = fg_bbox(rgb)

                    cv_rows.append({
                        'session': info['session'], 'task_type': info['task_type'],
                        'phase': ph, 'view': view,
                        'laplacian_var': lap, 'noise_std': ns,
                        'fft_high': fft, 'edge_density': edge,
                        'brightness': bri, 'saturation': sat_m, 'hue_entropy': hent,
                        'fg_cx': fx, 'fg_cy': fy,
                    })
                except Exception: continue

cv_df = pd.DataFrame(cv_rows)
print(f'CV records: {len(cv_df):,}')


In [ ]:

# ── 4-3: Noise / Texture / Frequency  (view x phase) ─────────────────────────
cv_df['view_task'] = cv_df['view'] + '/' + cv_df['task_type']
order = [f'{v}/{t}' for v in VIEWS for t in ['nic','sc']]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat,
                   ['laplacian_var','noise_std','fft_high','edge_density']):
    sns.boxplot(data=cv_df, x='view_task', y=col, hue='phase',
                order=order, ax=ax, showfliers=False)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)
    if ax != axes.flat[0]:
        legend = ax.get_legend()
        if legend: legend.remove()

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', title='phase')
plt.suptitle('Noise / Texture / Frequency  [Input Feature Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

# ── 4-4: HSV scalar features  (view x task) ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['brightness','saturation','hue_entropy']):
    sns.boxplot(data=cv_df, x='view', y=col, hue='task_type', ax=ax, showfliers=False)
    ax.set_title(col)

plt.suptitle('HSV Scalar Features  [Input Appearance Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# brightness distribution by phase (key: are approach/insert frames visually different?)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, view in zip(axes, VIEWS):
    sns.histplot(data=cv_df[cv_df['view']==view], x='brightness', hue='phase',
                 bins=25, stat='density', common_norm=False,
                 element='step', fill=False, ax=ax)
    ax.set_title(f'{view}  brightness by phase')
plt.suptitle('Brightness Distribution: approach vs insert  [Phase-domain Shift]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

# ── 4-5: Foreground bbox center distribution  [Input Spatial Bias] ──────────
fg = cv_df.dropna(subset=['fg_cx','fg_cy'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, view in zip(axes, VIEWS):
    sub = fg[fg['view']==view]
    for task, color, m in [('nic','#2196F3','o'),('sc','#FF5722','^')]:
        ts = sub[sub['task_type']==task]
        ax.scatter(ts['fg_cx'], ts['fg_cy'], alpha=0.3, s=8,
                   c=color, marker=m, label=task)
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.invert_yaxis()
    ax.axhline(0.5, color='gray', lw=0.5, ls='--')
    ax.axvline(0.5, color='gray', lw=0.5, ls='--')
    ax.set_title(f'{view}  foreground center')
    ax.set_xlabel('cx (norm)'); ax.set_ylabel('cy (norm)')
    ax.legend(markerscale=3)

plt.suptitle('Foreground Blob Center Distribution  [Input Spatial Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nForeground IQR (lower = more concentrated = more biased):')
print(f'  {"view":8s} {"task":5s}  cx_IQR  cy_IQR')
for (view, task), sub in fg.groupby(['view','task_type']):
    cx_iqr = sub['fg_cx'].quantile(0.75) - sub['fg_cx'].quantile(0.25)
    cy_iqr = sub['fg_cy'].quantile(0.75) - sub['fg_cy'].quantile(0.25)
    flag = 'BIASED' if cx_iqr < 0.10 and cy_iqr < 0.10 else ''
    print(f'  {view:8s} {task:5s}  {cx_iqr:.3f}   {cy_iqr:.3f}  {flag}')


In [ ]:

# ── 5-1: Cross-view pixel difference  (same frame, insert phase) ─────────────
diff_lc = np.zeros((RESIZE_H, RESIZE_W))
diff_lr = np.zeros((RESIZE_H, RESIZE_W))
n = 0
for sess_dir in tqdm(ALL_SESSIONS[:10], desc='Pixel diff'):
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    ins = [l for l in lines if json.loads(l)['phase']=='insert']
    for ln in ins[::max(1,len(ins)//10)][:10]:
        d = json.loads(ln)
        obs = d.get('observation', {})
        paths = {v: obs.get(f'{v}_image',{}).get('path') for v in VIEWS}
        if not all(paths.values()): continue
        try:
            lg = load_gray(Path(paths['left']))
            cg = load_gray(Path(paths['center']))
            rg = load_gray(Path(paths['right']))
            diff_lc += np.abs(lg - cg)
            diff_lr += np.abs(lg - rg)
            n += 1
        except: continue

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, dm, title in [(axes[0], diff_lc/n, 'left vs center'),
                      (axes[1], diff_lr/n, 'left vs right')]:
    im = ax.imshow(dm, cmap='hot')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{title}  mean diff = {dm.mean():.2f}')
    ax.axis('off')

plt.suptitle('Cross-view Pixel Difference  [View Diversity]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Frames used: {n}')


In [ ]:

# ── 5-2: Cross-view feature correlation ──────────────────────────────────────
# High correlation → views carry similar info → shared encoder assumption holds

# Pivot to wide: one row per (session, step)
cv_wide_rows = []
for sess_dir in ALL_SESSIONS[:12]:
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    ins = [json.loads(l) for l in lines if json.loads(l)['phase']=='insert']
    for d in ins[::max(1,len(ins)//10)][:10]:
        obs = d.get('observation', {})
        row = {'session': info['session'], 'step': d['step']}
        all_ok = True
        for view in VIEWS:
            k = f'{view}_image'
            if k not in obs: all_ok = False; break
            try:
                rgb  = load_rgb(Path(obs[k]['path']))
                gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
                hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
                row[f'{view}_brightness'] = float(hsv[:,:,2].mean()/255)
                row[f'{view}_laplacian']  = float(cv2.Laplacian(gray,cv2.CV_32F).var())
                row[f'{view}_fft']        = fft_high_ratio(gray.astype(np.uint8))
                row[f'{view}_edge']       = float((cv2.Canny(gray.astype(np.uint8),40,120)>0).mean())
            except: all_ok = False; break
        if all_ok: cv_wide_rows.append(row)

wide_df = pd.DataFrame(cv_wide_rows)

feat_names = ['brightness','laplacian','fft','edge']
fig, axes = plt.subplots(1, len(feat_names), figsize=(18, 4))
corr_out = {}
for ax, feat in zip(axes, feat_names):
    lc = f'left_{feat}'; cc = f'center_{feat}'; rc = f'right_{feat}'
    if not all(c in wide_df for c in [lc,cc,rc]): continue
    r_lc = wide_df[lc].corr(wide_df[cc])
    r_lr = wide_df[lc].corr(wide_df[rc])
    r_cr = wide_df[cc].corr(wide_df[rc])
    corr_out[feat] = {'L-C': r_lc, 'L-R': r_lr, 'C-R': r_cr}
    ax.scatter(wide_df[lc], wide_df[cc], alpha=0.5, s=8, label=f'L-C r={r_lc:.2f}')
    ax.scatter(wide_df[cc], wide_df[rc], alpha=0.5, s=8, marker='s', label=f'C-R r={r_cr:.2f}')
    ax.set_title(feat); ax.legend(fontsize=7)

plt.suptitle('Cross-view Feature Correlation  [Shared Encoder Validity]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nPearson r (r>0.8 = views carry similar info → shared encoder OK):')
display(pd.DataFrame(corr_out).T.round(3))


In [ ]:

# ── 6: Bias Summary ──────────────────────────────────────────────────────────
phase_cnt = steps_df['phase'].value_counts()
task_cnt  = steps_df['task_type'].value_counts()
ins = steps_df[steps_df['phase']=='insert']

print('='*65)
print('  Bias Summary')
print('='*65)

print('\n[1] PHASE IMBALANCE  (input frame distribution)')
for ph in ['approach','insert','stabilize']:
    c = phase_cnt.get(ph, 0)
    print(f'    {ph:12s}: {c:6d}  ({c/phase_cnt.sum()*100:.1f}%)')
print('    -> oversample approach or subsample insert uniformly')

print('\n[2] TASK TYPE IMBALANCE')
for t, c in task_cnt.items():
    print(f'    {t:6s}: {c:6d}  ({c/task_cnt.sum()*100:.1f}%)')
print('    -> balanced batch sampling 1:1')

print('\n[3] TIP ERROR  (output distribution)')
for task in ['nic','sc']:
    sub = ins[ins['task_type']==task]
    xe, ye = sub['tip_x_error'].mean()*1000, sub['tip_y_error'].mean()*1000
    xs, ys = sub['tip_x_error'].std()*1000,  sub['tip_y_error'].std()*1000
    sym = 'BIASED' if abs(xe)>xs*0.3 or abs(ye)>ys*0.3 else 'symmetric'
    print(f'    {task.upper()}: x={xe:+.2f}mm (std={xs:.2f})  y={ye:+.2f}mm (std={ys:.2f})  [{sym}]')
print('    -> add Gaussian offset at insert start to diversify error range')

print('\n[4] SPATIAL INPUT BIAS  (see variance heatmap)')
for (view, task), vm in sorted(var_maps.items()):
    c = spatial_conc(vm)
    flag = 'BIASED' if c < 0.15 else ('OK' if c > 0.30 else 'caution')
    print(f'    {task:4s} {view:7s}: {c:.3f}  [{flag}]')
print('    -> RandomCrop / Affine augmentation if BIASED')

print('\n[5] RECOMMENDATIONS')
recs = [
    '1. Oversample approach ~4x relative to insert',
    '2. Batch sample nic:sc = 1:1',
    '3. Add Gaussian XY offset (sigma~5mm) at insert start for diverse tip_error',
    '4. For spatially biased views: RandomCrop + AffineTransform augmentation',
    '5. Temporal shuffle within session to prevent z_offset overfitting',
]
for r in recs:
    print(f'   {r}')
